## Chapter 5 – Implementing Quantum-Safe Security

This notebook explores how modern cryptography is adapting to the arrival of quantum computing. The focus is on the mathematical foundations of post-quantum cryptography and the practical engineering trade-offs involved in deploying quantum-safe security.

The exercises begin with a series of lattice-based examples that build intuition for high-dimensional geometry, noise, and the Closest Vector Problem (CVP). Using the recurring characters Daisy and Donald, the notebook demonstrates how information that is easy to recover with private knowledge can become difficult to recover from public information alone.

Next, the notebook introduces `Learning With Errors (LWE)`, showing how carefully structured noise transforms an ordinary algebra problem into a much harder one. Additional experiments explore how search complexity grows with dimension and how distance relationships change in high-dimensional spaces.

The final section moves from toy examples to production cryptography. Using the `pqcrypto` library, the notebook performs a complete `ML-KEM` key establishment exchange, compares post-quantum and classical key sizes, and visualizes the network and infrastructure trade-offs associated with deploying quantum-safe cryptography at internet scale.

### Code 5-0: Generate Lattice Figures 5-1 and 5-2

This code generates the two lattice visualizations used in this chapter. Figure 5-1 shows how a valid lattice point can be shifted by injected noise, producing a nearby coordinate that is no longer exactly on the lattice. Figure 5-2 illustrates how the same lattice can be described using different bases, including a clean basis and a skewed basis. Together, these figures introduce the ideas of noise, lattice geometry, and alternative coordinate descriptions that form the foundation of modern lattice-based cryptography.

#### Figure 5-1 diagram generation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# === Figure 5-1: Noise Shifts the Message Off the Lattice ===
GRID, LATTICE = 8, 24                                # reference grid / real lattice spacing
valid, noisy = np.array([24., 24.]), np.array([34.5, 29.])

# Guard: the valid point must stay the nearest real lattice point
lat = np.stack(np.meshgrid(np.arange(0, 97, LATTICE), np.arange(0, 73, LATTICE)), -1).reshape(-1, 2)
assert np.allclose(lat[np.linalg.norm(lat - noisy, axis=1).argmin()], valid)

fig, ax = plt.subplots(figsize=(12.8, 7.2))          # 16:9

# Background: faint reference grid, then the real lattice points
gx, gy = np.arange(8, 73, GRID), np.arange(8, 49, GRID)
for x in gx: ax.axvline(x, color="gray", ls=(0, (4, 4)), lw=.9, alpha=.3)
for y in gy: ax.axhline(y, color="gray", ls=(0, (4, 4)), lw=.9, alpha=.3)
ax.scatter(*np.meshgrid(gx, gy), s=45, fc="#EEEEEE", ec="#AAAAAA", lw=.8, zorder=2)
ax.scatter(*np.meshgrid(np.arange(24, 73, LATTICE), np.arange(24, 49, LATTICE)),
           s=135, fc="white", ec="#555555", lw=1.3, zorder=3)

# Decoding region: every position inside this square rounds back to the valid point
ax.add_patch(Rectangle(valid - LATTICE / 2, LATTICE, LATTICE, fc="royalblue", alpha=.07,
                       ec="royalblue", lw=1.2, ls=(0, (2, 3)), zorder=1))

# Foreground: valid point (blue), transmitted point (red), squiggly noise between them
ax.scatter(*valid, s=310, c="royalblue", ec="black", lw=1.8, zorder=7)
ax.scatter(*noisy, s=330, c="crimson", ec="black", lw=1.8, zorder=8)
d, t = noisy - valid, np.linspace(0, 1, 200)
normal = np.array([-d[1], d[0]]) / np.linalg.norm(d)
path = valid + np.outer(t, d) + np.outer(1.1 * np.sin(6 * np.pi * t) * np.sin(np.pi * t), normal)
ax.plot(*path.T, c="black", lw=2.4, zorder=5)
ax.annotate("", xy=noisy, xytext=path[-10], zorder=6,
            arrowprops=dict(arrowstyle="-|>", color="black", lw=2.2, mutation_scale=18))

# Labels: (text, points at, label position, color, arrow gap at target)
for text, xy, pos, color, gap in [
        ("Valid lattice point", valid, (7, 16), "black", 16),
        ("Injected noise", path[105], (15, 41), "crimson", 6),
        ("Noisy transmitted\ncoordinate", noisy, (44, 41), "black", 18)]:
    ax.annotate(text, xy=xy, xytext=pos, fontsize=15, color=color, ha="left", va="center", zorder=10,
                bbox=dict(boxstyle="round,pad=.35", fc="white", ec=color, lw=1, alpha=.96),
                arrowprops=dict(arrowstyle="-|>", color=color, lw=1.6, mutation_scale=18,
                                shrinkA=4, shrinkB=gap))

# Axes: no frame, arrowed x/y axes, extra room at top and right
ax.spines[:].set_visible(False)
ax.set(xlim=(0, 78), ylim=(0, 54), xticks=np.arange(0, 73, 8), yticks=np.arange(0, 49, 8))
ax.tick_params(length=0, labelsize=14, pad=7)
arrow = dict(arrowstyle="-|>", color="black", lw=2.2, mutation_scale=24, shrinkA=0, shrinkB=0)
ax.annotate("", xy=(77, 0), xytext=(0, 0), arrowprops=arrow)
ax.annotate("", xy=(0, 53), xytext=(0, 0), arrowprops=arrow)
ax.text(78, -1.6, "$x$", fontsize=24, ha="right", va="top")
ax.text(-2, 53, "$y$", fontsize=24, ha="center", va="bottom")

plt.tight_layout(pad=1.2)
plt.savefig("figure_5_1_noise_lattice_xy.png", dpi=300, bbox_inches="tight")
plt.show()

#### Figure 5-2 diagram generation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

# === Code 5-0: Same Lattice, Different Bases (Figure 5-2) ===
POINTS = np.array([(x, y) for x in range(-3, 4) for y in range(-3, 4)])   # one lattice, shown twice
TARGET = np.array([3, 2])

# (title, subtitle, footer, basis rows b1/b2, route offset, label offsets for b1/b2)
VIEWS = [
    ("Clean basis", "Short, perpendicular vectors", "Private-key style view",
     np.array([[1, 0], [0, 1]]), (0.1, 0.28), [(.52, -.48), (-.62, .42)]),
    ("Skewed basis", "Longer, skewed vectors", "Public-key style view",
     np.array([[1, 2], [1, 1]]), (0.0, 0.0), [(-.5, .3), (.55, .42)]),
]

def arrow(ax, start, end, color, lw, dashed=False, z=8, gap=0):
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle="-|>", mutation_scale=17, lw=lw,
                                 ls=(0, (5, 4)) if dashed else "-", color=color,
                                 alpha=.82 if dashed else 1, shrinkA=0, shrinkB=gap, zorder=z))

def route(basis):
    """Steps to TARGET: first along b1, then along b2 (coefficients solved, not hand-placed)."""
    coeffs = np.linalg.solve(basis.T, TARGET)
    assert np.allclose(coeffs, coeffs.round()), "target is not a lattice point for this basis"
    b1_leg = coeffs[0].round() * basis[0]
    return [np.zeros(2), b1_leg, b1_leg + coeffs[1].round() * basis[1]]   # e.g. clean: 3·b1, 2·b2

fig, axes = plt.subplots(1, 2, figsize=(12, 5.6))
for ax, (title, subtitle, footer, basis, offset, label_offsets) in zip(axes, VIEWS):
    assert round(abs(np.linalg.det(basis))) == 1, "basis must describe the same lattice"

    # Lattice, axes, and the gold target
    ax.grid(True, ls=(0, (4, 4)), color="gray", lw=1, alpha=.22)
    ax.axhline(0, color="black", lw=1.8, zorder=2)
    ax.axvline(0, color="black", lw=1.8, zorder=2)
    ax.scatter(*POINTS.T, s=85, color="#D9D9D9", edgecolor="#555555", zorder=3)
    ax.scatter(*TARGET, s=320, color="#F2C300", edgecolor="black", lw=1.8, zorder=10)

    # Gray route (nudged off the axes where needed), then the two basis vectors
    stops = [p + offset for p in route(basis)[:-1]] + [TARGET + (offset[0], 0)]
    for i, (a, b) in enumerate(zip(stops[:-1], stops[1:])):
        arrow(ax, a, b, "dimgray", lw=2.3, dashed=True, z=7, gap=10 if i == len(stops) - 2 else 0)
    for vec, color, name, off in zip(basis, ["royalblue", "crimson"], ["$b_1$", "$b_2$"], label_offsets):
        arrow(ax, (0, 0), vec, color, lw=3.6, z=9)
        ax.text(*(vec + off), name, fontsize=22, color=color, fontweight="bold", ha="center",
                va="center", bbox=dict(fc="white", ec="none", alpha=.82, pad=.1), zorder=11)

    # Titles and frame
    for y, text, style in [(1.10, title, dict(fontsize=20, color="#1155CC")),
                           (1.04, subtitle, dict(fontsize=12)),
                           (-.12, footer, dict(fontsize=12, color="dimgray", style="italic"))]:
        ax.text(.5, y, text, transform=ax.transAxes, ha="center", **style)
    ax.set(xlim=(-3.75, 3.75), ylim=(-3.75, 3.75), aspect="equal",
           xticks=range(-3, 4), yticks=range(-3, 4))
    ax.tick_params(length=0, labelsize=12)
    ax.spines[:].set_visible(False)

plt.subplots_adjust(wspace=.42, top=.86, bottom=.18)
plt.savefig("figure_5_2_lattice_bases.png", dpi=300, bbox_inches="tight")
plt.show()

### Code 5-1: Public and Private Views of a Toy Lattice

This example introduces the core intuition behind lattice-based cryptography.
Daisy places a message on a lattice, adds a small amount of noise, and then
recovers the original point using information that only she possesses.
Donald sees only the public lattice and must solve a Closest Vector Problem
(CVP) by searching nearby lattice points. Compare the work performed by each
person and consider how that effort changes as lattices grow in dimension.

In [ ]:
import numpy as np
import pandas as pd

# === Code 5-1: Public and Private Views of a Toy Lattice ===

# Step 1: Define the private (clean) and public (skewed) bases.
private_basis = np.array([[1, 0], [0, 1]])   # Daisy's map
public_basis  = np.array([[1, 2], [1, 1]])   # Public map

# Step 2: Place a message on the lattice and add calibrated noise.
message_point     = np.array([3, 2])                 # The secret message
rng               = np.random.default_rng(seed=42)
noise             = rng.uniform(-0.45, 0.45, size=2) # Small random offset
transmitted_point = message_point + noise            # What Donald sees

# Step 3: Daisy rounds using her private basis; Donald tries the same shortcut.
coeffs          = np.linalg.solve(private_basis.T, transmitted_point)
daisy_recovered = np.round(coeffs) @ private_basis    # [3, 2]

coeffs       = np.linalg.solve(public_basis.T, transmitted_point)
donald_guess = np.round(coeffs) @ public_basis        # [4, 3]

# Step 4: Donald brute-forces the Closest Vector Problem (CVP).
def nearest_lattice_point(target, basis, span=6, verbose=False):

    # Define a search window around the target.
    coords = np.arange(-span, span + 1)

    # Generate candidate recipes within that window.
    recipes = np.array(np.meshgrid(coords, coords)).T.reshape(-1, 2)

    # Convert recipes into lattice points.
    points = recipes @ basis

    # Measure the distance to each candidate.
    distances = np.linalg.norm(points - target, axis=1)

    # Select the closest match.
    best = np.argmin(distances)

    if verbose:
        nearby = distances < 2.0
        for p, d in zip(points[nearby], distances[nearby]):
            print(f"  checked {p.tolist()} -> distance {d:.3f}")

    return points[best], recipes[best], distances[best], len(distances)

donald_point, _, donald_dist, iters = nearest_lattice_point(
    transmitted_point, public_basis, verbose=True
)

# Step 5: Compare outcomes.
print(f"\nMessage point (secret):          {message_point.tolist()}")
print(f"Transmitted point (Donald sees): {transmitted_point.round(3).tolist()}")

actor_rows = [
    {"Actor": "Daisy", "Information Available": "Private basis",
     "Method": "Round using the clean basis",
     "Result": daisy_recovered.astype(int).tolist(), "Work Required": "1 rounding step"},
    {"Actor": "Donald", "Information Available": "Public basis only",
     "Method": "Brute-force CVP search",
     "Result": donald_point.tolist(), "Work Required": f"{iters} candidate checks"},
]

pd.DataFrame(actor_rows)

### Code 5-2: CVP Search Cost Across Dimensions

This example compares Daisy's recovery process with Donald's
brute-force Closest Vector Problem (CVP) search as the number of
dimensions increases. Daisy's workload remains nearly constant because
she knows the noise that was added. Donald must examine an ever-growing
number of candidate lattice points. Watch how quickly the search space
expands and consider what happens when the lattice grows from a simple
toy example to the high-dimensional spaces used in modern cryptography.

In [ ]:
import itertools
import time
import numpy as np
import pandas as pd

# === Code 5-2: CVP Search Cost Across Dimensions ===

TIMEOUT_SECONDS = 10


def make_bases(dims, seed=42):
    """Private: the clean grid. Public: the same lattice, skewed by random row mixing."""
    rng = np.random.default_rng(seed)
    private, public = np.eye(dims, dtype=int), np.eye(dims, dtype=int)
    for _ in range(3 * dims):                        # adding one row to another keeps the lattice
        i, j = rng.choice(dims, 2, replace=False)
        public[i] += rng.integers(1, 3) * public[j]
    return private, public


def recover_with_private_basis(transmitted, private_basis):
    """Daisy rounds using her clean basis (as in Code 5-1)."""
    coeffs = np.linalg.solve(private_basis.T, transmitted)
    return (np.round(coeffs) @ private_basis).astype(int)


def brute_force_cvp(target, basis, span=3):
    """Donald checks every candidate point in a search window."""
    values = range(-span, span + 1)
    best_point, best_dist = None, float("inf")
    start = time.time()

    for recipe in itertools.product(values, repeat=basis.shape[0]):
        if time.time() - start > TIMEOUT_SECONDS:
            return best_point, time.time() - start, False

        point = np.array(recipe) @ basis
        dist = np.linalg.norm(point - target)
        if dist < best_dist:
            best_point, best_dist = point, dist

    return best_point, time.time() - start, True


dimensions, span = [2, 4, 6, 8, 10], 3
rows = []

for dims in dimensions:

    # Create the two views of one lattice, a message, and a noisy transmission.
    private_basis, public_basis = make_bases(dims)
    assert round(abs(np.linalg.det(public_basis))) == 1   # same lattice as the clean grid
    message = np.full(dims, 3)
    noise = np.random.default_rng(42).uniform(-0.45, 0.45, size=dims)
    transmitted = message + noise

    # Daisy's recovery path.
    t0 = time.time()
    recovered = recover_with_private_basis(transmitted, private_basis)
    recovery_time = time.time() - t0

    # Donald's brute-force search, using only the public basis.
    _, search_time, completed = brute_force_cvp(transmitted, public_basis, span)
    candidates = (2 * span + 1) ** dims

    rows.append({
        "Dimensions": dims,
        "Candidates": f"{candidates:,}",
        "Recovery (sec)": f"{recovery_time:.6f}",
        "Search (sec)": f"{search_time:.3f}",
        "Recovery correct": np.array_equal(recovered, message),
        "Search finished": completed,
    })

    print(f"dim={dims:2d} | candidates={candidates:>15,} | recovery={recovery_time:.6f}s "
          f"| search={search_time:.3f}s | finished={completed}")

pd.DataFrame(rows)

### Code 5-3: Learning With Errors in a Toy System

This example demonstrates the central idea behind Learning With Errors
(LWE). A secret value is combined with a small amount of carefully
chosen noise before being made public. An authorized user removes the
noise and recovers the correct secret, while an attacker attempts to
solve the same system without knowing the noise. Compare the results to
see how a small amount of noise can transform an ordinary algebra
problem into a much harder one.

In [ ]:
import numpy as np
import pandas as pd

# === Code 5-3: Learning With Errors in a Toy System ===

q      = 17                              # Small modulus; values wrap around like a clock.
A      = np.array([[2, 3], [1, 4]])      # Public matrix. Everyone can see this.
secret = np.array([3, 2])                # Daisy's secret. Donald never sees it.
noise  = np.array([1, -1])               # Small random errors, never published.

# LWE core: public result = matrix @ secret + small noise, mod q.
b = (A @ secret + noise) % q

def solve_as_exact(public_b):
    """Ordinary linear algebra: treat the public result as exact, then round."""
    return np.round(np.linalg.solve(A, public_b)).astype(int)

cases = [("Without noise", "Solve A @ x = b", (A @ secret) % q),
         ("With noise (LWE)", "Same solve, same rounding", b)]

rows = []
for case, method, public_b in cases:
    guess = solve_as_exact(public_b)
    rows.append({"Case": case, "Public b": public_b.tolist(), "Method": method,
                 "Recovered secret": guess.tolist(), "Correct": np.array_equal(guess, secret)})

print("Public matrix A:\n", A)
print("Public vector b:", b.tolist())
print("Private secret: ", secret.tolist(), "\n")
pd.DataFrame(rows)

### Code 5-4: Distance Concentration Across Dimensions

This example generates Figure 5-3 by measuring the distance from random
points to their nearest lattice neighbors as the number of dimensions
increases. In low dimensions, nearby and faraway points are easier to
distinguish. As dimensions grow, those distances become increasingly
similar and the distributions tighten. This effect helps build intuition
for why high-dimensional lattice geometry becomes more difficult to
navigate and why distance alone becomes less useful as a guide.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# === Code 5-4: Distance Concentration Across Dimensions ===

dims_list = [2, 4, 8, 16, 32, 64]

# Colorful for ebook, ordered by luminance for grayscale print.
colors = ["#0B3C5D",  # dark blue
          "#328CC1",  # medium blue
          "#2E7D32",  # green
          "#F9A825",  # amber
          "#E65100",  # orange
          "#8E1B1B"]  # dark red

n_lattice, n_queries, bins = 500, 2000, 50
rows = []

# HD aspect ratio, but less wide than 15 x 6.
fig, ax = plt.subplots(figsize=(12, 6.75))

for dims, color in zip(dims_list, colors):
    rng = np.random.default_rng(dims)
    lattice = rng.integers(-5, 6, size=(n_lattice, dims))
    queries = rng.uniform(-3, 3, size=(n_queries, dims))

    nearest = np.array([
        np.min(np.linalg.norm(lattice - q, axis=1))
        for q in queries
    ])

    mean, std = nearest.mean(), nearest.std()
    rows.append({
        "Dimensions": dims,
        "Mean distance": round(mean, 3),
        "Std deviation": round(std, 3),
        "Relative spread": round(std / mean, 3)
    })

    ax.hist(nearest, bins=bins, density=True, alpha=0.60,
            color=color, edgecolor="none", label=f"{dims}D")
    ax.axvline(mean, color=color, linestyle="--", linewidth=2)

ax.set_xlabel("Distance to nearest lattice point", fontsize=16)
ax.set_ylabel("Density", fontsize=16)
ax.tick_params(axis="both", labelsize=14)
ax.legend(
    title="Dimensions",
    fontsize=16,
    title_fontsize=14,
    frameon=True,
    loc="upper right"
)

plt.tight_layout()
plt.savefig("figure_5_3_distance_concentration.png",
            dpi=300, bbox_inches="tight")
plt.show()

pd.DataFrame(rows)

### Code 5-5: ML-KEM in Practice

This example performs a complete ML-KEM key establishment exchange using
the same lattice-based algorithm now being deployed across the internet.
A public/private keypair is generated, a shared secret is encapsulated,
and the receiving party recovers the same secret through decapsulation.
The example also compares ML-KEM key and ciphertext sizes with classical
algorithms and generates Figure 5-5 to visualize the engineering
trade-offs involved in post-quantum deployment.


**Note:** Run the package installation cell that follows before executing
this example. The pqcrypto library provides the NIST post-quantum
algorithms used throughout this section.

In [ ]:
# === Install dependency ===
!pip install pqcrypto --quiet

In [ ]:
from pqcrypto.kem import ml_kem_512, ml_kem_768, ml_kem_1024
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import StrMethodFormatter

# === Code 5-5: ML-KEM in Practice ===

# Step 1: Run a complete ML-KEM-768 key exchange.
pub, sec     = ml_kem_768.keygen()        # generate public/private keypair
ct, key_enc  = ml_kem_768.encaps(pub)     # encapsulate shared secret
key_dec      = ml_kem_768.decaps(sec, ct) # decapsulate shared secret

print("=== ML-KEM-768 Key Exchange ===")
print(f"Public key size:    {len(pub):>6} bytes")
print(f"Ciphertext size:    {len(ct):>6} bytes")
print(f"Shared secret size: {len(key_enc):>6} bytes")
print(f"Keys match:         {key_enc == key_dec}")

# Step 2: Collect sizes for all ML-KEM levels plus classical baselines.
def kem_sizes(mod):
    pk, _ = mod.keygen()
    ct, key = mod.encaps(pk)
    return len(pk), len(ct), len(key)

levels = [("ML-KEM-512", ml_kem_512),
          ("ML-KEM-768", ml_kem_768),
          ("ML-KEM-1024", ml_kem_1024)]

rows = [
    ("RSA-2048",  "Classical", 294, 256, 32, "No"), # approximate baseline
    ("ECC P-256", "Classical",  64,  96, 32, "No"),
    *[(name, "Post-quantum", *kem_sizes(mod), "Yes")
      for name, mod in levels]
]
cols = ["Algorithm", "Type", "Public key (B)", "Ciphertext (B)",
        "Shared key (B)", "Quantum safe"]
df = pd.DataFrame(rows, columns=cols)

print("\n=== Algorithm Comparison ===")
print(df.to_string(index=False))

# Step 3: Visualize byte overhead and connection throughput (Figure 5-4).
RED, BLUE, GREEN, GRAY = "#c62828", "#1565c0", "#2e7d32", "#666666"
BUDGET = 10_000_000  # 10 MB/s spent only on handshakes

def plot_comparison(df, save="figure_5_4_mlkem_overhead.png"):
    names  = df["Algorithm"].tolist()
    pub    = df["Public key (B)"].to_numpy()
    ct     = df["Ciphertext (B)"].to_numpy()
    totals = pub + ct
    rate   = BUDGET / totals
    x      = np.arange(len(names))
    colors = [RED if t == "Classical" else BLUE for t in df["Type"]]

    # 16:9 canvas with extra breathing room.
    fig, ax = plt.subplots(figsize=(16, 9))
    ax.bar(x, pub, .5, color=colors, alpha=.95)             # public key
    ax.bar(x, ct, .5, bottom=pub, color=colors, alpha=.40)  # ciphertext

    box = dict(boxstyle="round,pad=.25", fc="white",
               ec="#cccccc", alpha=.9)
    for xi, total in zip(x, totals):
        ax.text(xi, total + 55, f"{total:,} B", ha="center",
                fontsize=18, fontweight="bold", bbox=box)

    ax.axhline(1500, color=GRAY, ls="--", lw=1.5)
    ax.text(.01, 1535, "Typical packet size (1,500 B)",
            transform=ax.get_yaxis_transform(),
            color=GRAY, fontsize=18)
    ax.set_ylabel("Total bytes (key + ciphertext)", fontsize=18)
    ax.set_xticks(x, names, fontsize=18)
    ax.tick_params(axis="y", labelsize=18)
    ax.set_ylim(0, 3700)
    ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))

    # Right axis estimates connections/sec under a fixed bandwidth budget.
    ax2 = ax.twinx()
    ax2.plot(x, rate, "o--", color=GREEN, lw=2.5, ms=12)

    box = dict(boxstyle="round,pad=.3", fc="white",
               ec=GREEN, alpha=.9)
    for xi, val in zip(x, rate):
        ax2.annotate(f"{int(val):,}", (xi, val), xytext=(-15, 10),
                     textcoords="offset points", ha="right",
                     fontsize=18, fontweight="bold",
                     color=GREEN, bbox=box)

    ax2.set_ylim(0, 68_000)
    ax2.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
    ax2.set_ylabel("Connections per second\n(10 MB/s budget)",
                   fontsize=18, color=GREEN, labelpad=18)
    ax2.tick_params(axis="y", labelcolor=GREEN, labelsize=18)

    legend = [
        Patch(fc=RED, label="Classical"),
        Patch(fc=BLUE, label="Post-quantum"),
        Patch(fc=GRAY, alpha=.95, label="Public key bytes"),
        Patch(fc=GRAY, alpha=.40, label="Ciphertext bytes"),
        Line2D([0], [0], color=GREEN, marker="o", ls="--",
               lw=2.5, label="Connections/sec (right axis)")
    ]
    ax.legend(handles=legend, loc="upper center",
              bbox_to_anchor=(.5, -.08), ncol=5,
              fontsize=15, frameon=False)

    fig.subplots_adjust(left=.09, right=.88, top=.95, bottom=.17)
    plt.savefig(save, dpi=300, facecolor="white")
    plt.show()

plot_comparison(df)